# 06 · Stage-marker verification

Re-derives stage boundaries from LSL events and shows that the redundant **"100"** events
land right after a named stage-end (backup markers).  Also checks fixed stages are ~180 s.

In [1]:
import warnings, pathlib, sys
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("module://matplotlib_inline.backend_inline")
import matplotlib.pyplot as plt
_c = pathlib.Path.cwd()
REPO = next(p for p in [_c] + list(_c.parents) if (p / "analysis").is_dir())
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
from analysis import dataset, eegkit as K
DATA_DIR = None
recs = dataset.discover(DATA_DIR)
_gi, LABEL = {}, {}
for _r in recs:
    _g = _r["group"]; _gi[_g] = _gi.get(_g, 0) + 1
    LABEL[_r["key"]] = ("exp" if _g == "E" else "ctrl") + str(_gi[_g])
def lbl(r): return LABEL[r["key"]]
FIG = REPO / "reports" / "figures"; FIG.mkdir(parents=True, exist_ok=True)
def ec_window(r):
    st = K.get_stages(K.load(r["path"]), r["key"])
    return [s for s in st if s[0].startswith("rest_close")][0]
def get_iaf(r):
    raw = K.load(r["path"]); ec = ec_window(r)
    _, live = K.get(raw, ec[1], ec[2]); return K.iaf(raw, live, ec[1], ec[2])
def stage_of(r, nm):
    st = K.get_stages(K.load(r["path"]), r["key"]); return [s for s in st if s[0] == nm]
print(f"discovered {len(recs)} recordings ->", {lbl(r): r["group"] for r in recs})


discovered 4 recordings -> {'exp1': 'E', 'exp2': 'E', 'ctrl1': 'C', 'ctrl2': 'C'}


In [2]:
rows=[]
for r in recs:
    raw = K.load(r["path"]); ev = K.event_times(raw)
    hunds = [float(a["onset"]) for a in raw.annotations if "100" in str(a["description"])]
    ends = {k:v for k,v in ev.items() if k.endswith("_finish")}
    st = K.get_stages(raw, r["key"])
    offs = [h - min(ends.values(), key=lambda t: abs(t-h)) for h in hunds] if ends else []
    rows.append(dict(rec=lbl(r), group=r["group"], n100=len(hunds),
                     n_marker_stages=len(K.stages_from_markers(raw)),
                     max_offset_s=round(max(map(abs,offs)),2) if offs else np.nan))
    durs = [(n, round(b-a,1)) for n,a,b in st]
    print(f"{lbl(r)}  100-events={len(hunds)}  stage-durations:", durs)
print()
print(pd.DataFrame(rows).to_string(index=False))
print("\nAll '100' offsets <=1 s after a stage-end; fixed stages ~180 s (reading/speech are free).")


exp1  100-events=2  stage-durations: [('rest_open1', 180.0), ('rest_close1', 180.0), ('hypervent', 180.0), ('rest_open2', 180.0), ('reading', 182.4), ('rest_open3', 180.0), ('speech', 371.8), ('rest_open4', 180.0), ('rest_close4', 180.9)]


exp2  100-events=1  stage-durations: [('rest_open1', 180.0), ('rest_close1', 180.0), ('hypervent', 180.0), ('rest_open2', 180.0), ('reading', 180.0), ('speech', 347.5), ('rest_open4', 180.0), ('rest_close4', 181.0)]


ctrl1  100-events=0  stage-durations: [('rest_open1', 150.0), ('rest_close1', 190.0), ('hypervent', 180.0), ('rest_open2', 180.0), ('reading', 180.0), ('rest_open3', 180.0), ('speech', 265.0)]


ctrl2  100-events=1  stage-durations: [('speech', 373.5), ('rest_open4', 180.0), ('rest_close4', 181.0)]

  rec group  n100  n_marker_stages  max_offset_s
 exp1     E     2                9          1.03
 exp2     E     1                8          1.02
ctrl1     C     0                0           NaN
ctrl2     C     1                3          0.04

All '100' offsets <=1 s after a stage-end; fixed stages ~180 s (reading/speech are free).
